<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/03_Missingness_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# NOTEBOOK 03 — GOOGLE DRIVE / NOTEBOOK 02 ARTIFACT RECOVERY
# ============================================================

from pathlib import Path
import os

from google.colab import drive

print("=" * 100)
print("AIR-LLM — NOTEBOOK 03 DRIVE ARTIFACT RECOVERY")
print("=" * 100)


# ------------------------------------------------------------
# 1. Create a completely clean mountpoint
# ------------------------------------------------------------

CLEAN_MOUNT = Path("/content/air_llm_drive")

if CLEAN_MOUNT.exists():

    # Only remove the temporary mountpoint if it is empty.
    # Never delete project data here.
    if CLEAN_MOUNT.is_dir() and not any(CLEAN_MOUNT.iterdir()):
        pass
    else:
        raise RuntimeError(
            f"Temporary mountpoint is not empty:\n{CLEAN_MOUNT}\n"
            "Use another empty mountpoint."
        )
else:
    CLEAN_MOUNT.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 2. Mount Google Drive
# ------------------------------------------------------------

print("\nMOUNTING GOOGLE DRIVE")
print("-" * 100)
print(f"Mountpoint : {CLEAN_MOUNT}")

drive.mount(
    str(CLEAN_MOUNT),
    force_remount=False
)

print("Google Drive mounted successfully.")


# ------------------------------------------------------------
# 3. Define project root
# ------------------------------------------------------------

PROJECT_ROOT = (
    CLEAN_MOUNT
    / "MyDrive"
    / "AIR_LLM_Research"
)


print("\nPROJECT ROOT")
print("-" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Exists       : {PROJECT_ROOT.exists()}")
print(f"Is directory : {PROJECT_ROOT.is_dir()}")


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "AIR-LLM project root was not found on Google Drive:\n"
        f"{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# 4. Notebook 02 directories
# ------------------------------------------------------------

NOTEBOOK_02_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_02"
)

SPLIT_DIR = (
    NOTEBOOK_02_DIR
    / "splits"
)

METADATA_DIR = (
    NOTEBOOK_02_DIR
    / "metadata"
)

PROFILE_DIR = (
    NOTEBOOK_02_DIR
    / "profiles"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_02_manifest.json"
)


# ------------------------------------------------------------
# 5. Verify directories
# ------------------------------------------------------------

print("\nNOTEBOOK 02 DIRECTORIES")
print("-" * 100)

for name, path in {

    "Notebook 02":
        NOTEBOOK_02_DIR,

    "Splits":
        SPLIT_DIR,

    "Metadata":
        METADATA_DIR,

    "Profiles":
        PROFILE_DIR,

    "Manifest":
        MANIFEST_PATH

}.items():

    print(
        f"{name:15s} : "
        f"{'FOUND' if path.exists() else 'MISSING'} | "
        f"{path}"
    )


# ------------------------------------------------------------
# 6. Discover split files
# ------------------------------------------------------------

print("\nNOTEBOOK 02 SPLIT FILE DISCOVERY")
print("-" * 100)

if SPLIT_DIR.exists():

    split_files = sorted(
        SPLIT_DIR.rglob("*.csv")
    )

    print(
        f"CSV files found : "
        f"{len(split_files)}"
    )

    for path in split_files:

        print(
            f"  FOUND : {path}"
        )

else:

    split_files = []

    print(
        "SPLIT DIRECTORY NOT FOUND"
    )


# ------------------------------------------------------------
# 7. Required split files
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

REQUIRED_SPLITS = []

for dataset_id in DATASETS:

    for split_name in [
        "train",
        "validation",
        "test"
    ]:

        REQUIRED_SPLITS.append(
            SPLIT_DIR
            / dataset_id
            / f"{dataset_id}_{split_name}.csv"
        )


missing_splits = [
    path
    for path in REQUIRED_SPLITS
    if not path.exists()
]

empty_splits = [
    path
    for path in REQUIRED_SPLITS
    if path.exists()
    and path.stat().st_size == 0
]


print("\nREQUIRED NOTEBOOK 02 SPLITS")
print("-" * 100)

print(
    f"Required split files : "
    f"{len(REQUIRED_SPLITS)}"
)

print(
    f"Missing split files  : "
    f"{len(missing_splits)}"
)

print(
    f"Empty split files    : "
    f"{len(empty_splits)}"
)


if missing_splits:

    print("\nMISSING SPLITS:")

    for path in missing_splits:
        print(
            f"  MISSING : {path}"
        )


if empty_splits:

    print("\nEMPTY SPLITS:")

    for path in empty_splits:
        print(
            f"  EMPTY : {path}"
        )


# ------------------------------------------------------------
# 8. Final decision
# ------------------------------------------------------------

if missing_splits or empty_splits:

    print("\n" + "=" * 100)
    print(
        "NOTEBOOK 02 ARTIFACTS ARE NOT AVAILABLE "
        "AT THE EXPECTED GOOGLE DRIVE LOCATION"
    )
    print("=" * 100)

    raise RuntimeError(
        "Notebook 03 cannot begin because the persisted "
        "Notebook 02 split datasets were not found."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 02 ARTIFACT RECOVERY SUCCESSFUL")
print("=" * 100)

print(
    "All 9 experimental split files are available."
)

print(
    f"Split directory : {SPLIT_DIR}"
)

print(
    "Notebook 03 may now begin."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 03 DRIVE ARTIFACT RECOVERY

MOUNTING GOOGLE DRIVE
----------------------------------------------------------------------------------------------------
Mountpoint : /content/air_llm_drive
Mounted at /content/air_llm_drive
Google Drive mounted successfully.

PROJECT ROOT
----------------------------------------------------------------------------------------------------
Project root : /content/air_llm_drive/MyDrive/AIR_LLM_Research
Exists       : True
Is directory : True

NOTEBOOK 02 DIRECTORIES
----------------------------------------------------------------------------------------------------
Notebook 02     : FOUND | /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_02
Splits          : FOUND | /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_02/splits
Metadata        : FOUND | /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_02/metadata
Profiles        : FOUND | /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_02/p

In [5]:
# ============================================================
# NOTEBOOK 03.0 — ENVIRONMENT AND CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import shutil
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 03")
print("MISSINGNESS GENERATION AND EXPERIMENTAL SCENARIO PREPARATION")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/air_llm_drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "AIR-LLM project root not found:\n"
        f"{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# Notebook 02 input
# ------------------------------------------------------------

NOTEBOOK_02_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_02"
)

SPLIT_DIR = (
    NOTEBOOK_02_DIR
    / "splits"
)


# ------------------------------------------------------------
# Notebook 03 output
# ------------------------------------------------------------

NOTEBOOK_03_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_03"
)

SCENARIO_DIR = (
    NOTEBOOK_03_DIR
    / "scenarios"
)

MASK_DIR = (
    NOTEBOOK_03_DIR
    / "masks"
)

GROUND_TRUTH_DIR = (
    NOTEBOOK_03_DIR
    / "ground_truth"
)

DIAGNOSTIC_DIR = (
    NOTEBOOK_03_DIR
    / "diagnostics"
)

METADATA_DIR = (
    NOTEBOOK_03_DIR
    / "metadata"
)

PROFILE_DIR = (
    NOTEBOOK_03_DIR
    / "profiles"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_03_manifest.json"
)


for directory in [
    NOTEBOOK_03_DIR,
    SCENARIO_DIR,
    MASK_DIR,
    GROUND_TRUTH_DIR,
    DIAGNOSTIC_DIR,
    METADATA_DIR,
    PROFILE_DIR,
    MANIFEST_PATH.parent
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Experimental configuration
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

MISSINGNESS_MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5

MASTER_SEED = 42

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}


print("\nPROJECT PATHS")
print("-" * 100)

print(
    f"Project root       : {PROJECT_ROOT}"
)

print(
    f"Notebook 02 input  : {SPLIT_DIR}"
)

print(
    f"Notebook 03 output : {NOTEBOOK_03_DIR}"
)

print(
    f"Master seed        : {MASTER_SEED}"
)

print(
    f"Missingness rates  : {MISSINGNESS_RATES}"
)

print(
    f"Repetitions        : {REPETITIONS}"
)

print(
    f"Mechanisms         : {MISSINGNESS_MECHANISMS}"
)


# ------------------------------------------------------------
# Verify Notebook 02 inputs
# ------------------------------------------------------------

required_input_files = []

for dataset_id in DATASETS:

    required_input_files.append(
        SPLIT_DIR
        / dataset_id
        / f"{dataset_id}_train.csv"
    )


missing_inputs = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_inputs:

    raise FileNotFoundError(
        "Notebook 02 training data are unavailable:\n"
        + "\n".join(
            str(path)
            for path in missing_inputs
        )
    )


print("\nNotebook 02 training datasets verified.")

AIR-LLM — NOTEBOOK 03
MISSINGNESS GENERATION AND EXPERIMENTAL SCENARIO PREPARATION

PROJECT PATHS
----------------------------------------------------------------------------------------------------
Project root       : /content/air_llm_drive/MyDrive/AIR_LLM_Research
Notebook 02 input  : /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_02/splits
Notebook 03 output : /content/air_llm_drive/MyDrive/AIR_LLM_Research/data/notebook_03
Master seed        : 42
Missingness rates  : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions        : 5
Mechanisms         : ['MCAR', 'MAR', 'MNAR_APPROXIMATION']

Notebook 02 training datasets verified.


In [6]:
# ============================================================
# CELL 03.1 — LOAD TRAINING DATA
# ============================================================

TRAINING_DATASETS = {}

TRAINING_LOAD_ROWS = []

for dataset_id in DATASETS:

    path = (
        SPLIT_DIR
        / dataset_id
        / f"{dataset_id}_train.csv"
    )

    df = pd.read_csv(
        path,
        low_memory=False
    )

    if df.empty:

        raise ValueError(
            f"Training dataset is empty: {dataset_id}"
        )

    TRAINING_DATASETS[
        dataset_id
    ] = df

    TRAINING_LOAD_ROWS.append({

        "dataset_id":
            dataset_id,

        "path":
            str(path),

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "missing_cells":
            int(df.isna().sum().sum()),

        "target":
            TARGET_REGISTRY[dataset_id]

    })


TRAINING_LOAD_DF = pd.DataFrame(
    TRAINING_LOAD_ROWS
)

display(
    TRAINING_LOAD_DF
)

print(
    "Notebook 02 training datasets loaded successfully."
)

,dataset_id,path,rows,columns,missing_cells,target
0,adult_income,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,22792,16,2963,income
1,bank_marketing,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,31647,18,0,y
2,diabetes_130us,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,71236,49,261834,readmitted


Notebook 02 training datasets loaded successfully.


In [7]:
# ============================================================
# CELL 03.2 — PRESERVE GROUND TRUTH
# ============================================================

GROUND_TRUTH_DATASETS = {}

GROUND_TRUTH_HASHES = {}

GROUND_TRUTH_ROWS = []

for dataset_id in DATASETS:

    df = TRAINING_DATASETS[
        dataset_id
    ].copy(deep=True)

    GROUND_TRUTH_DATASETS[
        dataset_id
    ] = df

    hash_value = hashlib.sha256(
        pd.util.hash_pandas_object(
            df,
            index=True
        ).values.tobytes()
    ).hexdigest()

    GROUND_TRUTH_HASHES[
        dataset_id
    ] = hash_value

    GROUND_TRUTH_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "missing_cells":
            int(df.isna().sum().sum()),

        "sha256":
            hash_value

    })


GROUND_TRUTH_PROFILE_DF = pd.DataFrame(
    GROUND_TRUTH_ROWS
)

display(
    GROUND_TRUTH_PROFILE_DF
)

print(
    "Original training observations preserved in memory."
)

,dataset_id,rows,columns,missing_cells,sha256
0,adult_income,22792,16,2963,94aa3b8744263dd8e27b4ae9b10cab934c16f654447edd...
1,bank_marketing,31647,18,0,573dbd142de81aea86218869c09c6c7a088187b94c73f6...
2,diabetes_130us,71236,49,261834,9301d2e126c17b549eda7aadea2e67538c91fe54559618...


Original training observations preserved in memory.


In [8]:
# ============================================================
# CELL 03.3 — IDENTIFY MASKABLE FEATURES
# ============================================================

MASKING_REGISTRY = {}

MASKING_ROWS = []


def detect_identifier_like_columns(df):

    candidates = []

    for column in df.columns:

        series = df[column]

        non_null = series.dropna()

        if len(non_null) == 0:
            continue

        unique_ratio = (
            non_null.nunique()
            / len(non_null)
        )

        name_lower = str(column).lower()

        name_signal = any(
            token in name_lower
            for token in [
                "id",
                "identifier",
                "encounter_id",
                "patient_id",
                "customer_id",
                "record_id"
            ]
        )

        if (
            unique_ratio >= 0.98
            and name_signal
        ):

            candidates.append(column)

    return candidates


for dataset_id in DATASETS:

    df = TRAINING_DATASETS[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    identifier_candidates = (
        detect_identifier_like_columns(df)
    )

    constant_columns = [
        column
        for column in df.columns
        if df[column].nunique(
            dropna=False
        ) <= 1
    ]

    excluded = set(
        [target]
        + identifier_candidates
        + constant_columns
    )

    maskable_features = [
        column
        for column in df.columns
        if column not in excluded
    ]

    MASKING_REGISTRY[
        dataset_id
    ] = {

        "target":
            target,

        "identifier_columns":
            identifier_candidates,

        "constant_columns":
            constant_columns,

        "excluded_columns":
            sorted(excluded),

        "maskable_features":
            maskable_features

    }

    MASKING_ROWS.append({

        "dataset_id":
            dataset_id,

        "total_columns":
            len(df.columns),

        "target":
            target,

        "identifier_count":
            len(identifier_candidates),

        "constant_count":
            len(constant_columns),

        "excluded_count":
            len(excluded),

        "maskable_feature_count":
            len(maskable_features)

    })


MASKING_PROFILE_DF = pd.DataFrame(
    MASKING_ROWS
)

display(
    MASKING_PROFILE_DF
)

,dataset_id,total_columns,target,identifier_count,constant_count,excluded_count,maskable_feature_count
0,adult_income,16,income,1,0,2,14
1,bank_marketing,18,y,1,0,2,16
2,diabetes_130us,49,readmitted,1,3,5,44


In [9]:
# ============================================================
# CELL 03.4 — MISSINGNESS GENERATION UTILITIES
# ============================================================

def deterministic_seed(
    dataset_id,
    mechanism,
    rate,
    repetition
):

    key = (
        f"{MASTER_SEED}|"
        f"{dataset_id}|"
        f"{mechanism}|"
        f"{rate:.4f}|"
        f"{repetition}"
    )

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()

    return int(
        digest[:8],
        16
    )


def observed_mask(df, feature):

    return (
        df[feature].notna()
    )


def numeric_signal(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    median = numeric.median()

    if pd.isna(median):
        median = 0.0

    numeric = numeric.fillna(
        median
    )

    values = numeric.to_numpy(
        dtype=float
    )

    if np.nanstd(values) == 0:

        return np.zeros(
            len(values),
            dtype=float
        )

    values = (
        values - np.nanmean(values)
    ) / (
        np.nanstd(values) + 1e-12
    )

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return values


def categorical_signal(series):

    values = series.astype(
        "string"
    )

    frequencies = (
        values.value_counts(
            normalize=True,
            dropna=False
        )
    )

    signal = (
        values.map(
            frequencies
        )
        .fillna(
            0.0
        )
        .to_numpy(
            dtype=float
        )
    )

    if np.nanstd(signal) == 0:

        return np.zeros(
            len(series),
            dtype=float
        )

    signal = (
        signal - np.mean(signal)
    ) / (
        np.std(signal) + 1e-12
    )

    return signal


def feature_signal(series):

    if pd.api.types.is_numeric_dtype(
        series
    ):

        return numeric_signal(
            series
        )

    return categorical_signal(
        series
    )


def choose_mar_drivers(
    df,
    target_feature,
    maskable_features
):

    candidates = [
        column
        for column in maskable_features
        if column != target_feature
    ]

    if not candidates:
        return []

    # Deterministic selection based on
    # column order rather than random state.
    selected = candidates[:3]

    return selected


def select_exact_rate(
    eligible_positions,
    propensity,
    rate,
    rng
):

    n_eligible = len(
        eligible_positions
    )

    if n_eligible == 0:
        return np.array([], dtype=int)

    desired = int(
        round(
            n_eligible * rate
        )
    )

    desired = min(
        desired,
        n_eligible
    )

    if desired <= 0:
        return np.array([], dtype=int)

    # Small random tie-breaker preserves
    # deterministic but non-identical selection.
    jitter = rng.random(
        n_eligible
    ) * 1e-10

    ranking = (
        np.asarray(propensity)
        + jitter
    )

    order = np.argsort(
        ranking
    )

    selected_local = order[
        -desired:
    ]

    return np.asarray(
        eligible_positions
    )[selected_local]


def build_empty_mask(df):

    return pd.DataFrame(
        False,
        index=df.index,
        columns=df.columns
    )


print(
    "Missingness generation utilities initialized."
)

Missingness generation utilities initialized.


In [10]:
# ============================================================
# CELL 03.5 — MCAR GENERATION
# ============================================================

def generate_mcar_mask(
    df,
    maskable_features,
    rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    mask = build_empty_mask(
        df
    )

    for feature in maskable_features:

        eligible = np.flatnonzero(
            df[feature].notna().to_numpy()
        )

        if len(eligible) == 0:
            continue

        desired = int(
            round(
                len(eligible) * rate
            )
        )

        desired = min(
            desired,
            len(eligible)
        )

        if desired <= 0:
            continue

        selected = rng.choice(
            eligible,
            size=desired,
            replace=False
        )

        mask.iloc[
            selected,
            mask.columns.get_loc(feature)
        ] = True

    return mask


print(
    "MCAR generator ready."
)

MCAR generator ready.


In [11]:
# ============================================================
# CELL 03.6 — MAR GENERATION
# ============================================================

def generate_mar_mask(
    df,
    maskable_features,
    rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    mask = build_empty_mask(
        df
    )

    for feature in maskable_features:

        eligible = np.flatnonzero(
            df[feature].notna().to_numpy()
        )

        if len(eligible) == 0:
            continue

        drivers = choose_mar_drivers(
            df,
            feature,
            maskable_features
        )

        drivers = [
            driver
            for driver in drivers
            if driver != feature
        ]

        if not drivers:

            propensity = np.zeros(
                len(df),
                dtype=float
            )

        else:

            driver_signals = []

            for driver in drivers:

                signal = feature_signal(
                    df[driver]
                )

                driver_signals.append(
                    signal
                )

            propensity = np.mean(
                np.vstack(
                    driver_signals
                ),
                axis=0
            )

        eligible_propensity = (
            propensity[eligible]
        )

        selected = select_exact_rate(
            eligible_positions=eligible,
            propensity=eligible_propensity,
            rate=rate,
            rng=rng
        )

        if len(selected) > 0:

            mask.iloc[
                selected,
                mask.columns.get_loc(feature)
            ] = True

    return mask


print(
    "MAR generator ready."
)

MAR generator ready.


In [12]:
# ============================================================
# CELL 03.7 — MNAR APPROXIMATION
# ============================================================

def generate_mnar_approximation_mask(
    df,
    maskable_features,
    rate,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    mask = build_empty_mask(
        df
    )

    for feature in maskable_features:

        eligible = np.flatnonzero(
            df[feature].notna().to_numpy()
        )

        if len(eligible) == 0:
            continue

        signal = feature_signal(
            df[feature]
        )

        eligible_signal = (
            signal[eligible]
        )

        selected = select_exact_rate(
            eligible_positions=eligible,
            propensity=eligible_signal,
            rate=rate,
            rng=rng
        )

        if len(selected) > 0:

            mask.iloc[
                selected,
                mask.columns.get_loc(feature)
            ] = True

    return mask


print(
    "MNAR approximation generator ready."
)

MNAR approximation generator ready.


In [13]:
# ============================================================
# CELL 03.8 — MISSINGNESS RATE UTILITIES
# ============================================================

def calculate_mask_rate(
    original_df,
    mask,
    maskable_features
):

    eligible_cells = 0
    masked_cells = 0

    for feature in maskable_features:

        eligible = (
            original_df[feature].notna()
        )

        eligible_count = int(
            eligible.sum()
        )

        masked_count = int(
            mask.loc[
                eligible,
                feature
            ].sum()
        )

        eligible_cells += (
            eligible_count
        )

        masked_cells += (
            masked_count
        )

    if eligible_cells == 0:

        return 0.0

    return (
        masked_cells
        / eligible_cells
    )


print(
    "Missingness-rate utilities ready."
)

Missingness-rate utilities ready.


In [14]:
# ============================================================
# CELL 03.9 — REPEATED MASKING
# ============================================================

SCENARIO_REGISTRY = []

MASK_OBJECTS = {}

MASKED_DATA_OBJECTS = {}

GROUND_TRUTH_OBJECTS = {}

for dataset_id in DATASETS:

    df = GROUND_TRUTH_DATASETS[
        dataset_id
    ]

    maskable_features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    for mechanism in MISSINGNESS_MECHANISMS:

        for rate in MISSINGNESS_RATES:

            for repetition in range(
                1,
                REPETITIONS + 1
            ):

                seed = deterministic_seed(
                    dataset_id,
                    mechanism,
                    rate,
                    repetition
                )

                if mechanism == "MCAR":

                    mask = generate_mcar_mask(
                        df,
                        maskable_features,
                        rate,
                        seed
                    )

                elif mechanism == "MAR":

                    mask = generate_mar_mask(
                        df,
                        maskable_features,
                        rate,
                        seed
                    )

                elif mechanism == "MNAR_APPROXIMATION":

                    mask = (
                        generate_mnar_approximation_mask(
                            df,
                            maskable_features,
                            rate,
                            seed
                        )
                    )

                else:

                    raise ValueError(
                        f"Unknown mechanism: {mechanism}"
                    )

                scenario_id = (
                    f"{dataset_id}"
                    f"__{mechanism}"
                    f"__rate_{int(rate * 100):02d}"
                    f"__rep_{repetition:02d}"
                )

                masked_df = df.copy(
                    deep=True
                )

                # Preserve only newly masked cells.
                for feature in maskable_features:

                    masked_df.loc[
                        mask[feature],
                        feature
                    ] = np.nan

                MASK_OBJECTS[
                    scenario_id
                ] = mask

                MASKED_DATA_OBJECTS[
                    scenario_id
                ] = masked_df

                GROUND_TRUTH_OBJECTS[
                    scenario_id
                ] = df.copy(
                    deep=True
                )

                actual_rate = calculate_mask_rate(
                    df,
                    mask,
                    maskable_features
                )

                SCENARIO_REGISTRY.append({

                    "scenario_id":
                        scenario_id,

                    "dataset_id":
                        dataset_id,

                    "mechanism":
                        mechanism,

                    "mechanism_class":
                        (
                            "MNAR_APPROXIMATION"
                            if mechanism
                            == "MNAR_APPROXIMATION"
                            else mechanism
                        ),

                    "requested_rate":
                        rate,

                    "actual_rate":
                        actual_rate,

                    "repetition":
                        repetition,

                    "seed":
                        seed,

                    "rows":
                        len(df),

                    "columns":
                        len(df.columns),

                    "maskable_features":
                        len(maskable_features),

                    "masked_cells":
                        int(mask.sum().sum()),

                    "original_missing_cells":
                        int(df.isna().sum().sum())

                })


SCENARIO_REGISTRY_DF = pd.DataFrame(
    SCENARIO_REGISTRY
)

print(
    f"Generated scenarios : "
    f"{len(SCENARIO_REGISTRY_DF)}"
)

display(
    SCENARIO_REGISTRY_DF.head(10)
)

Generated scenarios : 225


,scenario_id,dataset_id,mechanism,mechanism_class,requested_rate,actual_rate,repetition,seed,rows,columns,maskable_features,masked_cells,original_missing_cells
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,MCAR,0.1,0.099992,1,3920181198,22792,16,14,31610,2963
1,adult_income__MCAR__rate_10__rep_02,adult_income,MCAR,MCAR,0.1,0.099992,2,3719379823,22792,16,14,31610,2963
2,adult_income__MCAR__rate_10__rep_03,adult_income,MCAR,MCAR,0.1,0.099992,3,224306034,22792,16,14,31610,2963
3,adult_income__MCAR__rate_10__rep_04,adult_income,MCAR,MCAR,0.1,0.099992,4,1692051507,22792,16,14,31610,2963
4,adult_income__MCAR__rate_10__rep_05,adult_income,MCAR,MCAR,0.1,0.099992,5,3366700048,22792,16,14,31610,2963
5,adult_income__MCAR__rate_20__rep_01,adult_income,MCAR,MCAR,0.2,0.199984,1,2767163,22792,16,14,63220,2963
6,adult_income__MCAR__rate_20__rep_02,adult_income,MCAR,MCAR,0.2,0.199984,2,2740139601,22792,16,14,63220,2963
7,adult_income__MCAR__rate_20__rep_03,adult_income,MCAR,MCAR,0.2,0.199984,3,282536738,22792,16,14,63220,2963
8,adult_income__MCAR__rate_20__rep_04,adult_income,MCAR,MCAR,0.2,0.199984,4,1164937861,22792,16,14,63220,2963
9,adult_income__MCAR__rate_20__rep_05,adult_income,MCAR,MCAR,0.2,0.199984,5,3663661479,22792,16,14,63220,2963


In [15]:
# ============================================================
# CELL 03.10 — MASK REPRODUCIBILITY
# ============================================================

REPRODUCIBILITY_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    seed = row[
        "seed"
    ]

    df = GROUND_TRUTH_DATASETS[
        dataset_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    if mechanism == "MCAR":

        regenerated = generate_mcar_mask(
            df,
            features,
            rate,
            seed
        )

    elif mechanism == "MAR":

        regenerated = generate_mar_mask(
            df,
            features,
            rate,
            seed
        )

    else:

        regenerated = (
            generate_mnar_approximation_mask(
                df,
                features,
                rate,
                seed
            )
        )

    original = MASK_OBJECTS[
        scenario_id
    ]

    identical = (
        original.equals(
            regenerated
        )
    )

    REPRODUCIBILITY_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "mechanism":
            mechanism,

        "rate":
            rate,

        "repetition":
            repetition,

        "seed":
            seed,

        "mask_reproducible":
            identical

    })


MASK_REPRODUCIBILITY_DF = pd.DataFrame(
    REPRODUCIBILITY_ROWS
)

display(
    MASK_REPRODUCIBILITY_DF
)

if not MASK_REPRODUCIBILITY_DF[
    "mask_reproducible"
].all():

    raise RuntimeError(
        "Mask reproducibility validation failed."
    )

print(
    "Mask reproducibility: PASSED"
)

,scenario_id,dataset_id,mechanism,rate,repetition,seed,mask_reproducible
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,3920181198,True
1,adult_income__MCAR__rate_10__rep_02,adult_income,MCAR,0.1,2,3719379823,True
2,adult_income__MCAR__rate_10__rep_03,adult_income,MCAR,0.1,3,224306034,True
3,adult_income__MCAR__rate_10__rep_04,adult_income,MCAR,0.1,4,1692051507,True
4,adult_income__MCAR__rate_10__rep_05,adult_income,MCAR,0.1,5,3366700048,True
...,...,...,...,...,...,...,...
220,diabetes_130us__MNAR_APPROXIMATION__rate_50__r...,diabetes_130us,MNAR_APPROXIMATION,0.5,1,3275598522,True
221,diabetes_130us__MNAR_APPROXIMATION__rate_50__r...,diabetes_130us,MNAR_APPROXIMATION,0.5,2,3235126846,True
222,diabetes_130us__MNAR_APPROXIMATION__rate_50__r...,diabetes_130us,MNAR_APPROXIMATION,0.5,3,1400253057,True
223,diabetes_130us__MNAR_APPROXIMATION__rate_50__r...,diabetes_130us,MNAR_APPROXIMATION,0.5,4,404539835,True


Mask reproducibility: PASSED


In [16]:
# ============================================================
# CELL 03.11 — GROUND-TRUTH STORAGE
# ============================================================

GROUND_TRUTH_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    ground_truth = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    scenario_ground_truth_dir = (
        GROUND_TRUTH_DIR
        / dataset_id
    )

    scenario_ground_truth_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    ground_truth_path = (
        scenario_ground_truth_dir
        / f"{scenario_id}_ground_truth.csv"
    )

    mask_path = (
        MASK_DIR
        / dataset_id
    )

    mask_path.mkdir(
        parents=True,
        exist_ok=True
    )

    mask_file = (
        mask_path
        / f"{scenario_id}_mask.csv"
    )

    ground_truth.to_csv(
        ground_truth_path,
        index=False
    )

    mask.astype(
        "int8"
    ).to_csv(
        mask_file,
        index=False
    )

    GROUND_TRUTH_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "ground_truth_path":
            str(ground_truth_path),

        "mask_path":
            str(mask_file),

        "ground_truth_size_bytes":
            ground_truth_path.stat().st_size,

        "mask_size_bytes":
            mask_file.stat().st_size

    })


GROUND_TRUTH_STORAGE_DF = pd.DataFrame(
    GROUND_TRUTH_ROWS
)

print(
    f"Ground-truth artifacts saved : "
    f"{len(GROUND_TRUTH_STORAGE_DF)}"
)

Ground-truth artifacts saved : 225


In [17]:
# ============================================================
# CELL 03.12 — SAVE MASKED EXPERIMENTAL DATA
# ============================================================

SCENARIO_STORAGE_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    masked_df = MASKED_DATA_OBJECTS[
        scenario_id
    ]

    dataset_scenario_dir = (
        SCENARIO_DIR
        / dataset_id
    )

    dataset_scenario_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    scenario_path = (
        dataset_scenario_dir
        / f"{scenario_id}.csv"
    )

    masked_df.to_csv(
        scenario_path,
        index=False
    )

    SCENARIO_STORAGE_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "scenario_path":
            str(scenario_path),

        "rows":
            len(masked_df),

        "columns":
            len(masked_df.columns),

        "size_bytes":
            scenario_path.stat().st_size

    })


SCENARIO_STORAGE_DF = pd.DataFrame(
    SCENARIO_STORAGE_ROWS
)

print(
    f"Masked scenarios saved : "
    f"{len(SCENARIO_STORAGE_DF)}"
)

Masked scenarios saved : 225


In [18]:
# ============================================================
# CELL 03.13 — FEATURE MISSING COUNTS
# ============================================================

FEATURE_MISSING_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    original = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    for feature in features:

        original_missing = int(
            original[feature]
            .isna()
            .sum()
        )

        newly_masked = int(
            mask[feature]
            .sum()
        )

        total_missing = (
            original_missing
            + newly_masked
        )

        FEATURE_MISSING_ROWS.append({

            "scenario_id":
                scenario_id,

            "dataset_id":
                dataset_id,

            "mechanism":
                mechanism,

            "rate":
                rate,

            "repetition":
                repetition,

            "feature":
                feature,

            "original_missing":
                original_missing,

            "newly_masked":
                newly_masked,

            "total_missing":
                total_missing

        })


FEATURE_MISSING_DF = pd.DataFrame(
    FEATURE_MISSING_ROWS
)

display(
    FEATURE_MISSING_DF.head(20)
)

FEATURE_MISSING_DF.to_csv(
    DIAGNOSTIC_DIR
    / "feature_missing_counts.csv",
    index=False
)

,scenario_id,dataset_id,mechanism,rate,repetition,feature,original_missing,newly_masked,total_missing
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,0,2279,2279
1,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,workclass,1275,2152,3427
2,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,fnlwgt,0,2279,2279
3,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,education,0,2279,2279
4,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,education_num,0,2279,2279
5,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,marital_status,0,2279,2279
6,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,occupation,1281,2151,3432
7,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,relationship,0,2279,2279
8,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,race,0,2279,2279
9,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,sex,0,2279,2279


In [19]:
# ============================================================
# CELL 03.14 — MISSING PERCENTAGES
# ============================================================

MISSING_PERCENTAGE_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    original = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    for feature in features:

        eligible = int(
            original[feature]
            .notna()
            .sum()
        )

        masked = int(
            mask[feature]
            .sum()
        )

        percentage = (
            masked / eligible * 100
            if eligible > 0
            else 0.0
        )

        MISSING_PERCENTAGE_ROWS.append({

            "scenario_id":
                scenario_id,

            "dataset_id":
                dataset_id,

            "mechanism":
                mechanism,

            "requested_rate":
                rate,

            "repetition":
                repetition,

            "feature":
                feature,

            "eligible_observations":
                eligible,

            "masked_observations":
                masked,

            "missing_percentage":
                percentage

        })


MISSING_PERCENTAGE_DF = pd.DataFrame(
    MISSING_PERCENTAGE_ROWS
)

MISSING_PERCENTAGE_DF.to_csv(
    DIAGNOSTIC_DIR
    / "missing_percentages.csv",
    index=False
)

display(
    MISSING_PERCENTAGE_DF.head(20)
)

,scenario_id,dataset_id,mechanism,requested_rate,repetition,feature,eligible_observations,masked_observations,missing_percentage
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,22792,2279,9.999122
1,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,workclass,21517,2152,10.001394
2,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,fnlwgt,22792,2279,9.999122
3,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,education,22792,2279,9.999122
4,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,education_num,22792,2279,9.999122
5,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,marital_status,22792,2279,9.999122
6,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,occupation,21511,2151,9.999535
7,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,relationship,22792,2279,9.999122
8,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,race,22792,2279,9.999122
9,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,sex,22792,2279,9.999122


In [20]:
# ============================================================
# CELL 03.15 — ROW-LEVEL MISSINGNESS
# ============================================================

ROW_MISSINGNESS_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    row_counts = (
        mask[features]
        .sum(axis=1)
    )

    for index, count in row_counts.items():

        ROW_MISSINGNESS_ROWS.append({

            "scenario_id":
                scenario_id,

            "dataset_id":
                dataset_id,

            "mechanism":
                mechanism,

            "rate":
                rate,

            "repetition":
                repetition,

            "row_index":
                index,

            "masked_feature_count":
                int(count),

            "row_has_missing":
                bool(count > 0)

        })


ROW_MISSINGNESS_DF = pd.DataFrame(
    ROW_MISSINGNESS_ROWS
)

ROW_MISSINGNESS_DF.to_csv(
    DIAGNOSTIC_DIR
    / "row_level_missingness.csv",
    index=False
)

display(
    ROW_MISSINGNESS_DF.head(20)
)

,scenario_id,dataset_id,mechanism,rate,repetition,row_index,masked_feature_count,row_has_missing
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,0,1,True
1,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,1,0,False
2,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,2,3,True
3,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,3,2,True
4,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,4,0,False
5,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,5,1,True
6,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,6,0,False
7,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,7,1,True
8,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,8,1,True
9,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,9,1,True


In [22]:
# ============================================================
# CELL 03.16 — MISSINGNESS ASSOCIATIONS
# ============================================================

ASSOCIATION_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    indicator = (
        mask[features]
        .astype(float)
    )

    # To control computation on high-dimensional data,
    # calculate associations only among features
    # that were actually masked.
    active_features = [
        feature
        for feature in features
        if indicator[feature].sum() > 0
    ]

    if len(active_features) >= 2:

        corr = indicator[
            active_features
        ].corr()

        for i, feature_a in enumerate(
            active_features
        ):

            for feature_b in active_features[
                i + 1:
            ]:

                value = corr.loc[
                    feature_a,
                    feature_b
                ]

                if pd.isna(value):
                    value = 0.0

                ASSOCIATION_ROWS.append({

                    "scenario_id":
                        scenario_id,

                    "dataset_id":
                        dataset_id,

                    "mechanism":
                        mechanism,

                    "rate":
                        rate,

                    "repetition":
                        repetition,

                    "feature_a":
                        feature_a,

                    "feature_b":
                        feature_b,

                    "association":
                        float(value)

                })


MISSINGNESS_ASSOCIATIONS_DF = pd.DataFrame(
    ASSOCIATION_ROWS
)

MISSINGNESS_ASSOCIATIONS_DF.to_csv(
    DIAGNOSTIC_DIR
    / "missingness_associations.csv",
    index=False
)

print(
    f"Association records : "
    f"{len(MISSINGNESS_ASSOCIATIONS_DF)}"
)

display(
    MISSINGNESS_ASSOCIATIONS_DF.head(20)
)

Association records : 86775


,scenario_id,dataset_id,mechanism,rate,repetition,feature_a,feature_b,association
0,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,workclass,-0.013595
1,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,fnlwgt,-0.000429
2,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,education,-0.007255
3,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,education_num,-0.003354
4,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,marital_status,-0.003842
5,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,occupation,-0.004043
6,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,relationship,0.003471
7,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,race,-0.007255
8,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,sex,-0.004329
9,adult_income__MCAR__rate_10__rep_01,adult_income,MCAR,0.1,1,age,capital_gain,0.007372


In [ ]:
# ============================================================
# CELL 03.17 — MISSINGNESS PATTERN ANALYSIS
# ============================================================

PATTERN_ROWS = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mechanism = row[
        "mechanism"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    pattern = (
        mask[features]
        .astype(int)
        .astype(str)
        .agg(
            "".join,
            axis=1
        )
    )

    counts = (
        pattern.value_counts()
    )

    for pattern_code, count in counts.items():

        PATTERN_ROWS.append({

            "scenario_id":
                scenario_id,

            "dataset_id":
                dataset_id,

            "mechanism":
                mechanism,

            "rate":
                rate,

            "repetition":
                repetition,

            "pattern":
                pattern_code,

            "row_count":
                int(count),

            "row_percentage":
                (
                    count
                    / len(pattern)
                    * 100
                )

        })


MISSINGNESS_PATTERN_DF = pd.DataFrame(
    PATTERN_ROWS
)

MISSINGNESS_PATTERN_DF.to_csv(
    DIAGNOSTIC_DIR
    / "missingness_patterns.csv",
    index=False
)

display(
    MISSINGNESS_PATTERN_DF.head(20)
)

In [ ]:
# ============================================================
# CELL 03.18 — MCAR DIAGNOSTICS
# ============================================================

MCAR_DIAGNOSTIC_ROWS = []

for row in SCENARIO_REGISTRY:

    if row["mechanism"] != "MCAR":
        continue

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    original = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    row_missing = (
        mask[features]
        .sum(axis=1)
    )

    eligible_rows = (
        row_missing.index
    )

    # Association between row missingness and
    # observed feature signals.
    feature_associations = []

    for feature in features:

        signal = feature_signal(
            original.loc[
                eligible_rows,
                feature
            ]
        )

        if np.std(signal) == 0:

            continue

        association = np.corrcoef(
            signal,
            row_missing.to_numpy(
                dtype=float
            )
        )[0, 1]

        if not np.isnan(association):

            feature_associations.append(
                abs(float(association))
            )

    mean_abs_association = (
        float(
            np.mean(
                feature_associations
            )
        )
        if feature_associations
        else 0.0
    )

    MCAR_DIAGNOSTIC_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "rate":
            rate,

        "repetition":
            repetition,

        "mean_abs_observed_association":
            mean_abs_association,

        "diagnostic_interpretation":
            "Lower association is more consistent with MCAR"

    })


MCAR_DIAGNOSTIC_DF = pd.DataFrame(
    MCAR_DIAGNOSTIC_ROWS
)

MCAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mcar_diagnostics.csv",
    index=False
)

display(
    MCAR_DIAGNOSTIC_DF
)

03.15 MISSINGNESS VALIDATION
Scenarios discovered: 135


In [ ]:
# ============================================================
# CELL 03.19 — MAR DIAGNOSTIC EVIDENCE
# ============================================================

MAR_DIAGNOSTIC_ROWS = []

for row in SCENARIO_REGISTRY:

    if row["mechanism"] != "MAR":
        continue

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    original = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    association_values = []

    for feature in features:

        missing_indicator = (
            mask[feature]
            .astype(float)
            .to_numpy()
        )

        drivers = choose_mar_drivers(
            original,
            feature,
            features
        )

        for driver in drivers:

            signal = feature_signal(
                original[driver]
            )

            if (
                np.std(signal) == 0
                or np.std(missing_indicator) == 0
            ):

                continue

            association = np.corrcoef(
                signal,
                missing_indicator
            )[0, 1]

            if not np.isnan(association):

                association_values.append(
                    abs(float(association))
                )

    mean_abs_association = (
        float(
            np.mean(
                association_values
            )
        )
        if association_values
        else 0.0
    )

    MAR_DIAGNOSTIC_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "rate":
            rate,

        "repetition":
            repetition,

        "mean_abs_driver_association":
            mean_abs_association,

        "diagnostic_interpretation":
            "Observed-driver association provides evidence "
            "consistent with MAR construction"

    })


MAR_DIAGNOSTIC_DF = pd.DataFrame(
    MAR_DIAGNOSTIC_ROWS
)

MAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mar_diagnostic_evidence.csv",
    index=False
)

display(
    MAR_DIAGNOSTIC_DF
)


------------------------------------------------------------------------------------------
Validating: adult_income
------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Validating: bank_marketing
------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Validating: diabetes_130us
------------------------------------------------------------------------------------------

MISSINGNESS VALIDATION COMPLETED
Scenarios validated: 135
Valid scenarios: 135
Invalid scenarios: 0


,dataset_id,scenario,mechanism,missing_rate,actual_rate,rate_difference,seed,shape_preserved,observed_values_preserved,changed_observed_values,target_preserved,scenario_valid,data_path,mask_path
0,adult_income,mcar_10pct_seed_42,MCAR,0.1,0.100001,7.317751e-07,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
1,adult_income,mcar_10pct_seed_123,MCAR,0.1,0.100001,7.317751e-07,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
2,adult_income,mcar_10pct_seed_2024,MCAR,0.1,0.100001,7.317751e-07,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
3,adult_income,mcar_20pct_seed_42,MCAR,0.2,0.200001,1.463550e-06,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
4,adult_income,mcar_20pct_seed_123,MCAR,0.2,0.200001,1.463550e-06,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
5,adult_income,mcar_20pct_seed_2024,MCAR,0.2,0.200001,1.463550e-06,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
6,adult_income,mcar_30pct_seed_42,MCAR,0.3,0.299999,1.463550e-06,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
7,adult_income,mcar_30pct_seed_123,MCAR,0.3,0.299999,1.463550e-06,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
8,adult_income,mcar_30pct_seed_2024,MCAR,0.3,0.299999,1.463550e-06,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
9,adult_income,mcar_40pct_seed_42,MCAR,0.4,0.399999,7.317751e-07,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...


In [ ]:
# ============================================================
# CELL 03.20 — MNAR DIAGNOSTIC EVIDENCE
# ============================================================

MNAR_DIAGNOSTIC_ROWS = []

for row in SCENARIO_REGISTRY:

    if row[
        "mechanism"
    ] != "MNAR_APPROXIMATION":

        continue

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    rate = row[
        "requested_rate"
    ]

    repetition = row[
        "repetition"
    ]

    original = (
        GROUND_TRUTH_OBJECTS[
            scenario_id
        ]
    )

    mask = MASK_OBJECTS[
        scenario_id
    ]

    features = (
        MASKING_REGISTRY[
            dataset_id
        ]["maskable_features"]
    )

    self_associations = []

    for feature in features:

        missing_indicator = (
            mask[feature]
            .astype(float)
            .to_numpy()
        )

        value_signal = feature_signal(
            original[feature]
        )

        if (
            np.std(missing_indicator) == 0
            or np.std(value_signal) == 0
        ):

            continue

        association = np.corrcoef(
            value_signal,
            missing_indicator
        )[0, 1]

        if not np.isnan(association):

            self_associations.append(
                abs(float(association))
            )

    mean_abs_self_association = (
        float(
            np.mean(
                self_associations
            )
        )
        if self_associations
        else 0.0
    )

    MNAR_DIAGNOSTIC_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "rate":
            rate,

        "repetition":
            repetition,

        "mean_abs_self_association":
            mean_abs_self_association,

        "mechanism_status":
            "MNAR_APPROXIMATION",

        "interpretation":
            "Missingness is deliberately conditioned "
            "on the feature's own observed value; "
            "this is a controlled MNAR approximation."

    })


MNAR_DIAGNOSTIC_DF = pd.DataFrame(
    MNAR_DIAGNOSTIC_ROWS
)

MNAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mnar_diagnostic_evidence.csv",
    index=False
)

display(
    MNAR_DIAGNOSTIC_DF
)

,dataset_id,mechanism,missing_rate,scenarios,mean_actual_rate,mean_rate_difference,all_valid
0,adult_income,MAR,0.1,3,0.101322,1.321586e-03,True
1,adult_income,MAR,0.2,3,0.202802,2.801723e-03,True
2,adult_income,MAR,0.3,3,0.304039,4.039155e-03,True
3,adult_income,MAR,0.4,3,0.405334,5.333909e-03,True
4,adult_income,MAR,0.5,3,0.507354,7.354340e-03,True
5,adult_income,MCAR,0.1,3,0.100001,7.317751e-07,True
6,adult_income,MCAR,0.2,3,0.200001,1.463550e-06,True
7,adult_income,MCAR,0.3,3,0.299999,1.463550e-06,True
8,adult_income,MCAR,0.4,3,0.399999,7.317751e-07,True
9,adult_income,MCAR,0.5,3,0.500000,0.000000e+00,True


In [ ]:
# ============================================================
# CELL 03.21 — NATURAL MISSINGNESS SUMMARY
# ============================================================

NATURAL_MISSING_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH_DATASETS[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for feature in df.columns:

        missing_count = int(
            df[feature]
            .isna()
            .sum()
        )

        total_count = len(df)

        missing_percentage = (
            missing_count
            / total_count
            * 100
            if total_count > 0
            else 0.0
        )

        NATURAL_MISSING_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "is_target":
                feature == target,

            "dtype":
                str(
                    df[feature].dtype
                ),

            "rows":
                total_count,

            "natural_missing_count":
                missing_count,

            "natural_missing_percentage":
                missing_percentage

        })


NATURAL_MISSINGNESS_DF = pd.DataFrame(
    NATURAL_MISSING_ROWS
)

NATURAL_MISSINGNESS_DF.to_csv(
    PROFILE_DIR
    / "natural_missingness_summary.csv",
    index=False
)

display(
    NATURAL_MISSINGNESS_DF.head(30)
)

VALIDATION RESULTS SAVED
Detailed validation:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation.csv

Validation summary:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation_summary.csv

All saved cell-wise scenarios passed ground-truth and missingness validation.


In [ ]:
# ============================================================
# CELL 03.22 — SCENARIO REGISTRY
# ============================================================

SCENARIO_REGISTRY_DF = (
    SCENARIO_REGISTRY_DF
    .merge(
        SCENARIO_STORAGE_DF[
            [
                "scenario_id",
                "scenario_path"
            ]
        ],
        on="scenario_id",
        how="left"
    )
    .merge(
        GROUND_TRUTH_STORAGE_DF[
            [
                "scenario_id",
                "ground_truth_path",
                "mask_path"
            ]
        ],
        on="scenario_id",
        how="left"
    )
)

SCENARIO_REGISTRY_PATH = (
    METADATA_DIR
    / "scenario_registry.csv"
)

SCENARIO_REGISTRY_DF.to_csv(
    SCENARIO_REGISTRY_PATH,
    index=False
)

display(
    SCENARIO_REGISTRY_DF.head(10)
)

print(
    f"Scenario registry saved:\n"
    f"{SCENARIO_REGISTRY_PATH}"
)

03.16 FINALIZING EXPERIMENTAL SCENARIOS
Cell-wise scenarios: 135
Feature-wise registry not found.

VERIFYING SCENARIO FILES
All referenced scenario files verified.
Validation records: 135

NOTEBOOK 03 — EXPERIMENTAL SCENARIOS FINALIZED
Cell-wise scenarios   : 135
Feature-wise scenarios : 0
Total scenarios        : 135

Scenario registry:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/scenario_registry.csv

Validation:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation.csv

Summary:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_scenario_summary.csv

Metadata:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_metadata.json

Notebook 03 experimental artifacts successfully finalized.


In [ ]:
# ============================================================
# CELL 03.23 — FINAL VALIDATION
# ============================================================

VALIDATION_ROWS = []

expected_scenarios = (
    len(DATASETS)
    * len(MISSINGNESS_MECHANISMS)
    * len(MISSINGNESS_RATES)
    * REPETITIONS
)


# ------------------------------------------------------------
# Scenario count
# ------------------------------------------------------------

scenario_count_valid = (
    len(SCENARIO_REGISTRY_DF)
    == expected_scenarios
)


# ------------------------------------------------------------
# Mechanisms
# ------------------------------------------------------------

mechanism_valid = (
    set(
        SCENARIO_REGISTRY_DF[
            "mechanism"
        ].unique()
    )
    == set(
        MISSINGNESS_MECHANISMS
    )
)


# ------------------------------------------------------------
# Rates
# ------------------------------------------------------------

rate_valid = (
    set(
        np.round(
            SCENARIO_REGISTRY_DF[
                "requested_rate"
            ].unique(),
            6
        )
    )
    == set(
        np.round(
            MISSINGNESS_RATES,
            6
        )
    )
)


# ------------------------------------------------------------
# Repetitions
# ------------------------------------------------------------

repetition_valid = (
    set(
        SCENARIO_REGISTRY_DF[
            "repetition"
        ].unique()
    )
    == set(
        range(
            1,
            REPETITIONS + 1
        )
    )
)


# ------------------------------------------------------------
# Actual missingness rates
# ------------------------------------------------------------

actual_rate_valid = (
    np.abs(
        SCENARIO_REGISTRY_DF[
            "actual_rate"
        ]
        -
        SCENARIO_REGISTRY_DF[
            "requested_rate"
        ]
    )
    <= (
        1.0
        /
        SCENARIO_REGISTRY_DF[
            "masked_cells"
        ].clip(
            lower=1
        )
    )
).all()


# ------------------------------------------------------------
# No target masking
# ------------------------------------------------------------

target_masking_rows = []

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    mask = MASK_OBJECTS[
        scenario_id
    ]

    target_masking_rows.append(
        int(
            mask[target].sum()
        )
    )


target_masking_valid = (
    max(
        target_masking_rows
    )
    == 0
)


# ------------------------------------------------------------
# Excluded feature masking
# ------------------------------------------------------------

excluded_masking_valid = True

for row in SCENARIO_REGISTRY:

    scenario_id = row[
        "scenario_id"
    ]

    dataset_id = row[
        "dataset_id"
    ]

    mask = MASK_OBJECTS[
        scenario_id
    ]

    excluded = (
        MASKING_REGISTRY[
            dataset_id
        ]["excluded_columns"]
    )

    for column in excluded:

        if column in mask.columns:

            if mask[column].sum() != 0:

                excluded_masking_valid = False


# ------------------------------------------------------------
# Original ground truth preservation
# ------------------------------------------------------------

ground_truth_preservation_valid = True

for dataset_id in DATASETS:

    before_hash = (
        GROUND_TRUTH_HASHES[
            dataset_id
        ]
    )

    after_hash = hashlib.sha256(
        pd.util.hash_pandas_object(
            GROUND_TRUTH_DATASETS[
                dataset_id
            ],
            index=True
        ).values.tobytes()
    ).hexdigest()

    if before_hash != after_hash:

        ground_truth_preservation_valid = False


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

reproducibility_valid = (
    MASK_REPRODUCIBILITY_DF[
        "mask_reproducible"
    ].all()
)


# ------------------------------------------------------------
# Scenario files
# ------------------------------------------------------------

scenario_files_valid = all(
    Path(path).exists()
    and Path(path).stat().st_size > 0
    for path
    in SCENARIO_REGISTRY_DF[
        "scenario_path"
    ]
)


# ------------------------------------------------------------
# Ground truth files
# ------------------------------------------------------------

ground_truth_files_valid = all(
    Path(path).exists()
    and Path(path).stat().st_size > 0
    for path
    in SCENARIO_REGISTRY_DF[
        "ground_truth_path"
    ]
)


# ------------------------------------------------------------
# Mask files
# ------------------------------------------------------------

mask_files_valid = all(
    Path(path).exists()
    and Path(path).stat().st_size > 0
    for path
    in SCENARIO_REGISTRY_DF[
        "mask_path"
    ]
)


# ------------------------------------------------------------
# Diagnostic files
# ------------------------------------------------------------

required_diagnostic_files = [

    DIAGNOSTIC_DIR
    / "feature_missing_counts.csv",

    DIAGNOSTIC_DIR
    / "missing_percentages.csv",

    DIAGNOSTIC_DIR
    / "row_level_missingness.csv",

    DIAGNOSTIC_DIR
    / "missingness_associations.csv",

    DIAGNOSTIC_DIR
    / "missingness_patterns.csv",

    DIAGNOSTIC_DIR
    / "mcar_diagnostics.csv",

    DIAGNOSTIC_DIR
    / "mar_diagnostic_evidence.csv",

    DIAGNOSTIC_DIR
    / "mnar_diagnostic_evidence.csv",

    PROFILE_DIR
    / "natural_missingness_summary.csv",

    SCENARIO_REGISTRY_PATH

]

diagnostic_files_valid = all(
    path.exists()
    and path.stat().st_size > 0
    for path
    in required_diagnostic_files
)


# ------------------------------------------------------------
# Build validation table
# ------------------------------------------------------------

VALIDATION_RESULTS = {

    "scenario_count":
        scenario_count_valid,

    "mechanisms":
        mechanism_valid,

    "missingness_rates":
        rate_valid,

    "repetitions":
        repetition_valid,

    "actual_rate_control":
        actual_rate_valid,

    "target_never_masked":
        target_masking_valid,

    "excluded_features_never_masked":
        excluded_masking_valid,

    "ground_truth_preserved":
        ground_truth_preservation_valid,

    "mask_reproducibility":
        reproducibility_valid,

    "scenario_files_saved":
        scenario_files_valid,

    "ground_truth_files_saved":
        ground_truth_files_valid,

    "mask_files_saved":
        mask_files_valid,

    "diagnostic_files_saved":
        diagnostic_files_valid

}


FINAL_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": key,
            "status": (
                "PASS"
                if value
                else "FAIL"
            )
        }
        for key, value
        in VALIDATION_RESULTS.items()
    ]
)

display(
    FINAL_VALIDATION_DF
)

failed_checks = [
    key
    for key, value
    in VALIDATION_RESULTS.items()
    if not value
]

if failed_checks:

    raise RuntimeError(
        "Notebook 03 final validation failed:\n"
        + "\n".join(
            failed_checks
        )
    )

print(
    "Notebook 03 final validation: PASSED"
)

FINAL NOTEBOOK 03 VERIFICATION

adult_income
  Scenario files : 45
  Mask files     : 45

bank_marketing
  Scenario files : 45
  Mask files     : 45

diabetes_130us
  Scenario files : 45
  Mask files     : 45

NOTEBOOK 03 COMPLETED SUCCESSFULLY
Experimental scenarios are now available for Notebook 04.


In [ ]:
# ============================================================
# CELL 03.19 — FINAL VALIDATION
# ============================================================

VALIDATION_ROWS = []

EXPECTED_SCENARIOS = (
    len(DATASETS)
    *
    len(MISSINGNESS_MECHANISMS)
    *
    len(MISSINGNESS_RATES)
    *
    REPETITIONS
)


# ------------------------------------------------------------
# Scenario count
# ------------------------------------------------------------

scenario_count_valid = (
    len(SCENARIO_REGISTRY_DF)
    ==
    EXPECTED_SCENARIOS
)


# ------------------------------------------------------------
# Validate every scenario
# ------------------------------------------------------------

for _, scenario in (
    SCENARIO_REGISTRY_DF.iterrows()
):

    dataset_id = scenario[
        "dataset_id"
    ]

    scenario_id = scenario[
        "scenario_id"
    ]

    df = TRAINING_DATASETS[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    feature_columns = [
        column
        for column in df.columns
        if column != target
    ]

    mask_path = (
        MASK_DIR
        / dataset_id
        / f"{scenario_id}.npz"
    )

    gt_path = (
        GROUND_TRUTH_DIR
        / dataset_id
        / f"{scenario_id}.csv"
    )

    mask_exists = (
        mask_path.exists()
        and
        mask_path.stat().st_size > 0
    )

    ground_truth_exists = (
        gt_path.exists()
        and
        gt_path.stat().st_size > 0
    )

    mask_shape_valid = False
    target_not_masked = True
    observed_only_masked = True
    masked_count = 0

    if mask_exists:

        mask_data = np.load(
            mask_path,
            allow_pickle=True
        )

        mask_array = mask_data[
            "mask"
        ].astype(bool)

        mask_shape_valid = (
            mask_array.shape
            ==
            (
                len(df),
                len(feature_columns)
            )
        )

        masked_count = int(
            mask_array.sum()
        )

        # Target cannot be masked because it is excluded.
        target_not_masked = (
            target not in
            list(mask_data["columns"])
        )

        for column_index, column in enumerate(
            feature_columns
        ):

            selected = mask_array[
                :,
                column_index
            ]

            observed = (
                df[column]
                .notna()
                .to_numpy()
            )

            if np.any(
                selected
                &
                ~observed
            ):

                observed_only_masked = False
                break

    validation_pass = all([

        mask_exists,

        ground_truth_exists,

        mask_shape_valid,

        target_not_masked,

        observed_only_masked,

        masked_count > 0
    ])

    VALIDATION_ROWS.append({

        "scenario_id":
            scenario_id,

        "dataset_id":
            dataset_id,

        "mechanism":
            scenario["mechanism"],

        "rate":
            scenario["missingness_rate"],

        "repetition":
            scenario["repetition"],

        "mask_exists":
            mask_exists,

        "ground_truth_exists":
            ground_truth_exists,

        "mask_shape_valid":
            mask_shape_valid,

        "target_not_masked":
            target_not_masked,

        "observed_only_masked":
            observed_only_masked,

        "masked_cells":
            masked_count,

        "validation_status":
            (
                "PASS"
                if validation_pass
                else "FAIL"
            )
    })


NOTEBOOK_03_SCENARIO_VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)

display(
    NOTEBOOK_03_SCENARIO_VALIDATION_DF.head(20)
)


overall_valid = (

    scenario_count_valid

    and

    (
        NOTEBOOK_03_SCENARIO_VALIDATION_DF[
            "validation_status"
        ] == "PASS"
    ).all()

    and

    REPRODUCIBILITY_DF[
        "reproducible"
    ].all()
)


FINAL_VALIDATION_DF = pd.DataFrame({

    "validation": [

        "scenario_count",
        "all_masks_saved",
        "all_ground_truth_saved",
        "all_masks_shape_valid",
        "target_never_masked",
        "only_observed_values_masked",
        "mask_reproducibility"
    ],

    "status": [

        scenario_count_valid,

        (
            NOTEBOOK_03_SCENARIO_VALIDATION_DF[
                "mask_exists"
            ].all()
        ),

        (
            NOTEBOOK_03_SCENARIO_VALIDATION_DF[
                "ground_truth_exists"
            ].all()
        ),

        (
            NOTEBOOK_03_SCENARIO_VALIDATION_DF[
                "mask_shape_valid"
            ].all()
        ),

        (
            NOTEBOOK_03_SCENARIO_VALIDATION_DF[
                "target_not_masked"
            ].all()
        ),

        (
            NOTEBOOK_03_SCENARIO_VALIDATION_DF[
                "observed_only_masked"
            ].all()
        ),

        REPRODUCIBILITY_DF[
            "reproducible"
        ].all()
    ]
})

FINAL_VALIDATION_DF[
    "result"
] = np.where(
    FINAL_VALIDATION_DF[
        "status"
    ],
    "PASS",
    "FAIL"
)

display(
    FINAL_VALIDATION_DF
)


if not overall_valid:

    failed = FINAL_VALIDATION_DF.loc[
        FINAL_VALIDATION_DF[
            "status"
        ] == False,
        "validation"
    ].tolist()

    raise RuntimeError(
        "Notebook 03 final validation failed:\n"
        + "\n".join(failed)
    )


FINAL_VALIDATION_DF.to_csv(
    DIAGNOSTIC_DIR
    / "notebook_03_final_validation.csv",
    index=False
)

print(
    "Notebook 03 final validation: PASSED"
)

In [ ]:
# ============================================================
# CELL 03.20 — DRIVE PERSISTENCE VALIDATION
# ============================================================

REQUIRED_NOTEBOOK_03_FILES = [

    DIAGNOSTIC_DIR
    / "feature_missing_counts.csv",

    DIAGNOSTIC_DIR
    / "feature_missing_percentages.csv",

    DIAGNOSTIC_DIR
    / "row_level_missingness.csv",

    DIAGNOSTIC_DIR
    / "missingness_associations.csv",

    DIAGNOSTIC_DIR
    / "missingness_patterns.csv",

    DIAGNOSTIC_DIR
    / "mcar_diagnostics.csv",

    DIAGNOSTIC_DIR
    / "mar_diagnostic_evidence.csv",

    DIAGNOSTIC_DIR
    / "mnar_diagnostic_evidence.csv",

    DIAGNOSTIC_DIR
    / "natural_missingness_summary.csv",

    DIAGNOSTIC_DIR
    / "notebook_03_final_validation.csv"
]


missing_files = [
    path
    for path in REQUIRED_NOTEBOOK_03_FILES
    if not path.exists()
]

empty_files = [
    path
    for path in REQUIRED_NOTEBOOK_03_FILES
    if path.exists()
    and path.stat().st_size == 0
]


# Add all scenario masks and ground truth files.
mask_files = list(
    MASK_DIR.rglob("*.npz")
)

ground_truth_files = list(
    GROUND_TRUTH_DIR.rglob("*.csv")
)


expected_scenario_files = (
    EXPECTED_SCENARIOS
)

scenario_files_valid = (
    len(mask_files)
    ==
    expected_scenario_files
    and
    len(ground_truth_files)
    ==
    expected_scenario_files
)


print("=" * 100)
print("AIR-LLM — NOTEBOOK 03 DRIVE PERSISTENCE VALIDATION")
print("=" * 100)

print(
    f"Expected scenarios        : "
    f"{EXPECTED_SCENARIOS}"
)

print(
    f"Mask files saved           : "
    f"{len(mask_files)}"
)

print(
    f"Ground-truth files saved  : "
    f"{len(ground_truth_files)}"
)

print(
    f"Required diagnostics      : "
    f"{len(REQUIRED_NOTEBOOK_03_FILES)}"
)

print(
    f"Missing diagnostics       : "
    f"{len(missing_files)}"
)

print(
    f"Empty diagnostics         : "
    f"{len(empty_files)}"
)


if missing_files:

    print("\nMISSING FILES:")

    for path in missing_files:
        print(
            f"  MISSING : {path}"
        )


if empty_files:

    print("\nEMPTY FILES:")

    for path in empty_files:
        print(
            f"  EMPTY : {path}"
        )


if (
    missing_files
    or empty_files
    or not scenario_files_valid
):

    raise RuntimeError(
        "Notebook 03 Drive persistence validation failed."
    )


print()
print(
    "GOOGLE DRIVE PERSISTENCE VALIDATION: PASSED"
)
print("=" * 100)

In [ ]:
# ============================================================
# CELL 03.21 — NOTEBOOK 03 MANIFEST
# ============================================================

manifest = {

    "notebook": "Notebook 03",

    "notebook_name":
        "Missingness Injection and Experimental Scenario Preparation",

    "project":
        "AIR-LLM Research Project",

    "project_code":
        "AIR-LLM",

    "configuration_version":
        "CONFIG-v1",

    "random_seed":
        RANDOM_SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "missingness_mechanisms":
        MISSINGNESS_MECHANISMS,

    "missingness_rates":
        MISSINGNESS_RATES,

    "repetitions":
        REPETITIONS,

    "expected_scenarios":
        EXPECTED_SCENARIOS,

    "actual_scenarios":
        len(
            SCENARIO_REGISTRY_DF
        ),

    "mnar_status":
        "CONTROLLED_APPROXIMATION",

    "mnar_identification_claim":
        False,

    "ground_truth_policy":
        "Original complete training observations preserved before masking",

    "target_masking":
        False,

    "natural_missing_values":
        "Preserved",

    "masking_policy":
        "Only currently observed feature cells are eligible for synthetic masking",

    "split_source":
        str(SPLIT_DIR),

    "output_directory":
        str(NOTEBOOK_03_DIR),

    "mask_directory":
        str(MASK_DIR),

    "ground_truth_directory":
        str(GROUND_TRUTH_DIR),

    "diagnostic_directory":
        str(DIAGNOSTIC_DIR),

    "validation_status":
        "PASSED",

    "reproducibility_status":
        "PASSED",

    "drive_persistence_status":
        "PASSED"
}


NOTEBOOK_03_MANIFEST_PATH = (
    MANIFEST_DIR
    / "notebook_03_manifest.json"
)


with open(
    NOTEBOOK_03_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2
    )


print(
    f"Notebook 03 manifest saved:\n"
    f"{NOTEBOOK_03_MANIFEST_PATH}"
)

In [ ]:
# ============================================================
# CELL 03.22 — FINAL NOTEBOOK STATUS
# ============================================================

print()
print("=" * 100)
print("AIR-LLM — NOTEBOOK 03 COMPLETE")
print("=" * 100)

print("\nPROJECT")
print("-" * 100)

print(
    f"Project root         : "
    f"{PROJECT_ROOT}"
)

print(
    f"Datasets             : "
    f"{len(DATASETS)}"
)

print(
    f"Random seed          : "
    f"{RANDOM_SEED}"
)


print("\nEXPERIMENTAL DESIGN")
print("-" * 100)

print(
    f"Mechanisms           : "
    f"{', '.join(MISSINGNESS_MECHANISMS)}"
)

print(
    f"Missingness rates    : "
    f"{', '.join(str(int(r * 100)) + '%' for r in MISSINGNESS_RATES)}"
)

print(
    f"Repetitions          : "
    f"{REPETITIONS}"
)

print(
    f"Total scenarios      : "
    f"{EXPECTED_SCENARIOS}"
)


print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASETS:

    df = TRAINING_DATASETS[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    feature_count = (
        len(df.columns)
        - 1
    )

    print(
        f"{dataset_id:20s} | "
        f"Rows: {len(df):8d} | "
        f"Features: {feature_count:3d} | "
        f"Target: {target}"
    )


print("\nMISSINGNESS METHODS")
print("-" * 100)

print(
    "MCAR                   : IMPLEMENTED"
)

print(
    "MAR                    : IMPLEMENTED"
)

print(
    "MNAR approximation     : IMPLEMENTED"
)

print(
    "MNAR identification    : NOT CLAIMED"
)


print("\nGROUND TRUTH")
print("-" * 100)

print(
    "Original training data : PRESERVED"
)

print(
    "Scenario ground truth  : SAVED"
)

print(
    "Target masking         : NO"
)

print(
    "Natural missingness    : PRESERVED"
)


print("\nMASKING")
print("-" * 100)

print(
    f"Scenario masks         : "
    f"{len(mask_files)}"
)

print(
    f"Ground-truth files     : "
    f"{len(ground_truth_files)}"
)

print(
    "Mask reproducibility   : VERIFIED"
)


print("\nDIAGNOSTICS")
print("-" * 100)

print(
    "Feature missing counts : READY"
)

print(
    "Missing percentages    : READY"
)

print(
    "Row-level missingness  : READY"
)

print(
    "Missingness associations : READY"
)

print(
    "Pattern analysis       : READY"
)

print(
    "MCAR diagnostics       : READY"
)

print(
    "MAR diagnostics        : READY"
)

print(
    "MNAR diagnostics       : READY"
)

print(
    "Natural missingness    : READY"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Scenario validation    : PASSED"
)

print(
    "Target protection      : PASSED"
)

print(
    "Observed-only masking  : PASSED"
)

print(
    "Reproducibility        : PASSED"
)

print(
    "Drive persistence      : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    f"Notebook 03 directory : "
    f"{NOTEBOOK_03_DIR}"
)

print(
    f"Mask directory         : "
    f"{MASK_DIR}"
)

print(
    f"Ground-truth directory : "
    f"{GROUND_TRUTH_DIR}"
)

print(
    f"Diagnostics directory  : "
    f"{DIAGNOSTIC_DIR}"
)

print(
    f"Manifest               : "
    f"{NOTEBOOK_03_MANIFEST_PATH}"
)


print("\nNEXT NOTEBOOK")
print("-" * 100)

print(
    "Notebook 04 — Imputation Baseline / Candidate Strategies"
)

print()
print("=" * 100)
print(
    "ALL NOTEBOOK 03 VALIDATIONS PASSED"
)
print(
    "AIR-LLM MISSINGNESS SCENARIOS ARE READY"
)
print(
    "GROUND TRUTH AND MASKS ARE PERSISTED"
)
print(
    "NOTEBOOK 04 IS AUTHORIZED TO BEGIN"
)
print("=" * 100)